# Benchmark

## classifiers f_small f_mid f_large

In [2]:
from pathlib import Path
import gc
import json
import platform
import statistics
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from DMTimeShardDataset import DMTimeShardDataset
from training import label_encoding
from training_models import models_htable

PROJECT_DIR = Path.cwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Working directory: {PROJECT_DIR}")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")

Working directory: /cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training
Device: cuda
PyTorch: 2.2.0a0+81ea7a4
GPU: NVIDIA A100-SXM4-40GB
CUDA runtime: 12.3


model config

In [ ]:
CONFIG = {
    "input_size": 256,
    "split": "train",
    "batch_sizes": [1],
    "num_warmup": 30,
    "num_repeats": 200,
    "data_path": Path("/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs"),
    "labels_path": Path("/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs/B0531+21_59000_48386_DM_time_dataset_realbased_labels_val.npy"),
    "output_dir": PROJECT_DIR / "benchmark_outputs",
    "models": {
        "f_small": {
            "model_name": "DM_time_binary_classificator_241002_3_GAP",
            "mode": "dmt",
            "weights_path": Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth"),
        },
        "f_mid": {
            "model_name": "DM_time_binary_classificator_241002_5_GAP",
            "mode": "ft",
            "weights_path": Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth"),
        },
        "f_large": {
            "model_name": "DM_time_binary_classificator_resnet18",
            "mode": "dmft",
            "weights_path": Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth"),
        },
    },
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
CONFIG

{'input_size': 256,
 'split': 'train',
 'batch_sizes': [1],
 'num_warmup': 30,
 'num_repeats': 200,
 'data_path': PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs'),
 'labels_path': PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs/B0531+21_59000_48386_DM_time_dataset_realbased_labels_val.npy'),
 'output_dir': PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/benchmark_outputs'),
 'models': {'f_small': {'model_name': 'DM_time_binary_classificator_241002_3_GAP',
   'mode': 'dmt',
   'weights_path': PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints_new/ch_point_DM_time_binary_classificator_241002_3_GAP_256/prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth')},
  'f_mid': {'model_name': 'DM_time_binary_classificator_241002_5_GAP',
   'mode': 'ft',
   'weights_path': PosixPath('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/b

In [4]:
#load models

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
    return checkpoint


def load_model(model_name, mode, weights_path, input_size=256):
    weights_path = Path(weights_path)
    if not weights_path.exists():
        raise FileNotFoundError(weights_path)

    model = models_htable[model_name](input_size, mode, False, device).to(device)
    checkpoint = torch.load(weights_path, map_location=device)
    model.load_state_dict(extract_state_dict(checkpoint))
    model.eval()
    return model


models = {}
for model_key, cfg in CONFIG["models"].items():
    print(f"Loading {model_key}: {cfg['model_name']} ({cfg['mode']})")
    models[model_key] = load_model(
        cfg["model_name"],
        cfg["mode"],
        cfg["weights_path"],
        input_size=CONFIG["input_size"],
    )

print("Loaded models:", list(models.keys()))

# preparing batch

def infer_prefix(labels_path):
    labels_name = Path(labels_path).name
    suffixes = [
        "_DM_time_dataset_realbased_labels_test.npy",
        "_DM_time_dataset_realbased_labels_train.npy",
        "_DM_time_dataset_realbased_labels_val.npy",
        "_DM_time_dataset_realbased_labels_joined.npy",
    ]
    for suffix in suffixes:
        if labels_name.endswith(suffix):
            return labels_name[: -len(suffix)]
    return Path(labels_name).stem


def move_batch_to_device(batch):
    return {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}


def make_batch_slice(batch, batch_size):
    sliced = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            sliced[key] = value[:batch_size]
        elif isinstance(value, (list, tuple)):
            sliced[key] = value[:batch_size]
        else:
            sliced[key] = value
    return sliced


dataset_cfg = {
    "output_dir": str(CONFIG["data_path"]),
    "prefix": infer_prefix(CONFIG["labels_path"]),
}

dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split=CONFIG["split"])
dataset.labels = label_encoding(dataset.labels.astype(object))

max_batch_size = max(CONFIG["batch_sizes"])
loader = DataLoader(dataset, batch_size=max_batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
base_batch = move_batch_to_device(next(iter(loader)))

print(f"Dataset split: {CONFIG['split']}")
print(f"Dataset samples: {len(dataset)}")
print("Batch tensor shapes:")
for key, value in base_batch.items():
    if torch.is_tensor(value):
        print(f"  {key}: {tuple(value.shape)} {value.dtype} {value.device}")
        


#time measurement

def _forward(model, batch):
    if hasattr(model, "forward"):
        return model.forward(batch)
    return model(batch)

def get_latency(model, batch, iters=100):
    model.eval()
    
    # Warmup
    for _ in range(10): 
        _forward(model, batch)
        
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start_event.record()
    for _ in range(iters):
        _forward(model, batch)
    end_event.record()
    torch.cuda.synchronize()
    
    return start_event.elapsed_time(end_event) / iters

# memory usage measurement gpu

def get_memory(model, batch):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _forward(model, batch)
    return torch.cuda.max_memory_allocated() / (1024**2)

def param_memory(model):
    total = sum(p.numel() * p.element_size() for p in model.parameters())
    return total / (1024**2)

Loading f_small: DM_time_binary_classificator_241002_3_GAP (dmt)
Loading f_mid: DM_time_binary_classificator_241002_5_GAP (ft)
Loading f_large: DM_time_binary_classificator_resnet18 (dmft)
Loading f_mid: DM_time_binary_classificator_241002_5_GAP (ft)
Loading f_large: DM_time_binary_classificator_resnet18 (dmft)
Loaded models: ['f_small', 'f_mid', 'f_large']
Loaded models: ['f_small', 'f_mid', 'f_large']
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Dataset split: train
Dataset samples: 1377792
Batch tensor shapes:
  dm_time: (1, 256, 256) torch.float32 cuda:0
  label: (1,) torch.int64 cuda:0
  metadata: (1, 5) torch.float32 cuda:0
  freq_time: (1, 256, 256) torch.float32 cuda:0
Dataset split: train
Dataset samples: 1377792
Batch tensor shapes:
  dm_time: (1, 256, 256) torch.float32 cuda:0
  label: (1,) torch.int64 cuda:0
  metadata: (1, 5) torch.float32 cuda:0
  freq_time: (1, 256, 256) torch.floa

benchmark models and embeddings

In [6]:
results = []
batch = make_batch_slice(base_batch, 1)

def measure_submodule(func_name, model, x_input, iters=100):
    if not hasattr(model, func_name):
        return None, None
        
    func = getattr(model, func_name)
    model.eval()
    
    # Warmup
    for _ in range(10): 
        func(x_input)
        
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    # Latency Messung
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    for _ in range(iters):
        func(x_input)
    end_event.record()
    torch.cuda.synchronize()
    
    latency = start_event.elapsed_time(end_event) / iters
    
    # Mem Messung
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    func(x_input)
    mem_peak = torch.cuda.max_memory_allocated() / (1024**2)
    
    return latency, mem_peak

def submodule_param_memory(model, func_name):
    if func_name == "classifier_features":
        modules = [m for n, m in model.named_children() if "conv" in n or "pool" in n and "gap" not in n]
    elif func_name == "pooled_features":
        modules = [m for n, m in model.named_children() if "conv" in n or "pool" in n]
    elif func_name == "classifier":
        modules = list(model.modules())
    else:
        return 0
        
    total = sum(p.numel() * p.element_size() for m in modules for p in m.parameters() if p.requires_grad)
    return total / (1024**2)


for key, model in models.items():
    # Gesamt-Inferenz via Forward (entspricht classifier())
    latency_full = get_latency(model, batch)
    mem_peak_full = get_memory(model, batch)
    mem_param_full = param_memory(model)
    
    results.append({
        "Model": key,
        "Sub-Function": "Full Forward",
        "Latency (ms)": f"{latency_full:.2f}",
        "Peak Mem (MB)": f"{mem_peak_full:.2f}", 
        "Param Mem (MB)": f"{mem_param_full:.2f}"
    })
    
    sub_funcs = ["classifier_features", "pooled_features"]
    
    # Führe `extract_model_inputs` korrekt aus, auch wenn es ein Tupel zurückgibt. 
    # Bei "dmft" (z.B. f_large_dmft) werden die kombinierten Kanäle benötigt. 
    # Bei den anderen z.b. dmt oder ft kommt nur (tensor,) als tuple zurück.
    try:
        extracted = model.extract_model_inputs(batch)
        if isinstance(extracted, tuple):
            if len(extracted) == 1:
                data_input = extracted[0]
            else:
                if hasattr(model, "process_inputs"):
                    data_input = model.process_inputs(*extracted)
                else: 
                    data_input = torch.cat(extracted, dim=1)
        else:
            data_input = extracted
    except AttributeError:
        tensor_keys = [k for k, v in batch.items() if torch.is_tensor(v)]
        if "data" in tensor_keys:
            data_input = batch["data"]
        else:
            data_input = batch[tensor_keys[0]]

    # Workaround falls shape immer noch nicht passt
    if hasattr(model, 'input_channels') and data_input.shape[1] != model.input_channels:
        # Erstelle Dummy-Daten mit korrekter anzahl an channels
        data_input = torch.rand((1, model.input_channels, 256, 256)).to(device)


    for sub in sub_funcs:
        lat, mem = measure_submodule(sub, model, data_input)
        if lat is not None:
             # Näherung für params
            mem_p = submodule_param_memory(model, sub)
            results.append({
                "Model": key,
                "Sub-Function": sub,
                "Latency (ms)": f"{lat:.2f}",
                "Peak Mem (MB)": f"{mem:.2f}", 
                "Param Mem (MB)": f"{mem_p:.4f}"
            })

pd.DataFrame(results)

,Model,Sub-Function,Latency (ms),Peak Mem (MB),Param Mem (MB)
0,f_small,Full Forward,0.36,57.62,0.02
1,f_small,classifier_features,0.30,57.33,0.0161
2,f_small,pooled_features,0.37,57.33,0.0161
3,f_mid,Full Forward,0.63,60.27,0.02
4,f_mid,classifier_features,0.49,59.41,0.0206
5,f_mid,pooled_features,0.44,59.41,0.0206
6,f_large,Full Forward,3.24,88.10,42.63
7,f_large,classifier_features,3.08,87.85,0.0000


# Dedispersion

In [4]:
import your
import tracemalloc

fil_path = "/cephfs/users/oleksjuk/MA/Andrei/B0531+21_59000_48386.fil"
fil = your.Your(fil_path)
hdr = fil.your_header
tsamp = float(hdr.tsamp)

def dedisperse_and_norm(data, dm, hdr):
    nchans, nsamp = data.shape
    freqs = float(hdr.fch1) + np.arange(nchans) * float(hdr.foff)
    # Delay in Samples
    delay_samp = np.rint(4.148808e3 * dm * (freqs**-2 - np.max(freqs)**-2) / tsamp).astype(int)
    
    dedisp = np.zeros_like(data, dtype=np.float32)
    for ch in range(nchans):
        shift = delay_samp[ch]
        if shift > 0:
            dedisp[ch, :-shift] = data[ch, shift:]
        elif shift < 0:
            dedisp[ch, -shift:] = data[ch, :shift]
        else:
            dedisp[ch] = data[ch]
        
    mn, mx = dedisp.min(), dedisp.max()
    if mx > mn:
        dedisp = (dedisp - mn) / (mx - mn) * 255
    return dedisp.astype(np.uint8)[:, :256]


meta = base_batch["metadata"][0] # (snr, t_min, t_max, DM, label)
t_min, t_max, dm = meta[1].item(), meta[2].item(), meta[3].item()

t_center = (t_min + t_max) / 2
start_samp = max(0, int(t_center / tsamp) - 128)


freqs = float(hdr.fch1) + np.arange(hdr.nchans) * float(hdr.foff)
max_delay = int(np.abs(np.rint(4.148808e3 * dm * (freqs**-2 - np.max(freqs)**-2) / tsamp)).max())

iters = 100
raw_data = fil.get_data(start_samp, 256 + max_delay).T

def measure(func):
    tracemalloc.start()
    t0 = time.perf_counter()
    for _ in range(iters):
        func()
    t1 = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return (t1 - t0) * 1000 / iters, peak / (1024**2)

pure_time, pure_mem = measure(lambda: dedisperse_and_norm(raw_data, dm, hdr))
full_time, full_mem = measure(lambda: dedisperse_and_norm(fil.get_data(start_samp, 256 + max_delay).T, dm, hdr))

results_dedisp = [
    {"Mode": "Reine Dedispersion & Norm", "Latency (ms)": f"{pure_time:.2f}", "Peak CPU Mem (MB)": f"{pure_mem:.2f}"},
    {"Mode": "Inkl. I/O (get_data)", "Latency (ms)": f"{full_time:.2f}", "Peak CPU Mem (MB)": f"{full_mem:.2f}"}
]

print(f"DM der Probe: {dm:.2f}")
pd.DataFrame(results_dedisp)

DM der Probe: 0.00


,Mode,Latency (ms),Peak CPU Mem (MB)
0,Reine Dedispersion & Norm,0.67,0.52
1,Inkl. I/O (get_data),0.70,0.57


In [7]:
import your
import tracemalloc
import time
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

fil_path = "/cephfs/users/oleksjuk/MA/Andrei/B0531+21_59000_48386.fil"
fil = your.Your(fil_path)
hdr = fil.your_header
tsamp = float(hdr.tsamp)

def dedisperse_and_norm_vectorized(data, dm, hdr):
    nchans, nsamp = data.shape
    freqs = float(hdr.fch1) + np.arange(nchans) * float(hdr.foff)
    
    # Delay in Samples
    delay_samp = np.rint(4.148808e3 * dm * (freqs**-2 - np.max(freqs)**-2) / tsamp).astype(int)
    
    row_idx = np.arange(nchans)[:, None]
    col_idx = np.arange(256)[None, :] + delay_samp[:, None]
    
    dedisp = data[row_idx, col_idx].astype(np.float32)
        
    mn, mx = dedisp.min(), dedisp.max()
    if mx > mn:
        dedisp = (dedisp - mn) / (mx - mn) * 255
    return dedisp.astype(np.uint8)


num_samples = 100
# load 100 random shuffled samples for benchmarking and averaging (DM!=0, DM==0)
rand_loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)
tasks = []
freqs = float(hdr.fch1) + np.arange(hdr.nchans) * float(hdr.foff)

# calculate max_delay for each target
for i, batch in enumerate(rand_loader):
    if i >= num_samples: 
        break
    meta = batch["metadata"][0]
    t_min, t_max, dm = meta[1].item(), meta[2].item(), meta[3].item()
    
    t_center = (t_min + t_max) / 2
    start_samp = max(0, int(t_center / tsamp) - 128)
    max_delay = int(np.abs(np.rint(4.148808e3 * dm * (freqs**-2 - np.max(freqs)**-2) / tsamp)).max())
    
    tasks.append({"dm": dm, "start_samp": start_samp, "max_delay": max_delay})

# preload raw filterbank slices 
raw_data_list = []
for t in tasks:
    raw_data_list.append(fil.get_data(t["start_samp"], 256 + t["max_delay"]).T)

def measure_pure():
    tracemalloc.start()
    t0 = time.perf_counter()
    for raw_data, t in zip(raw_data_list, tasks):
        dedisperse_and_norm_vectorized(raw_data, t["dm"], hdr)
    t1 = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return (t1 - t0) * 1000 / len(tasks), peak / (1024**2)

def measure_full():
    tracemalloc.start()
    t0 = time.perf_counter()
    for t in tasks:
        data = fil.get_data(t["start_samp"], 256 + t["max_delay"]).T
        dedisperse_and_norm_vectorized(data, t["dm"], hdr)
    t1 = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return (t1 - t0) * 1000 / len(tasks), peak / (1024**2)

pure_time, pure_mem = measure_pure()
full_time, full_mem = measure_full()

results_dedisp = [
    {"Mode": "Reine Dedispersion & Norm (Numpy Vec - Avg 100 Samples)", "Latency (ms)": f"{pure_time:.2f}", "Peak CPU Mem (MB)": f"{pure_mem:.2f}"},
    {"Mode": "Inkl. I/O (get_data) (Numpy Vec - Avg 100 Samples)", "Latency (ms)": f"{full_time:.2f}", "Peak CPU Mem (MB)": f"{full_mem:.2f}"}
]

pd.DataFrame(results_dedisp)

,Mode,Latency (ms),Peak CPU Mem (MB)
0,Reine Dedispersion & Norm (Numpy Vec - Avg 100...,0.46,1.01
1,Inkl. I/O (get_data) (Numpy Vec - Avg 100 Samp...,0.52,2.06


# embedding rejector r1

In [9]:
from embedding_processing_models import build_embedding_processing

rejector_checkpoints = {
    "conv_mlp": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_3_GAP_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker2_trial3-042-0.712-0.635.pth",
        "kwargs": {"model_name": "conv_mlp", "cnn_channels": 64, "extra_conv": False, "pool_size": 7, "hidden_dim": 64}
    },
    "spatial_pool_mlp": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-run_embedding_3_GAP_spatial_pool_mlp_lr1.47e-05_wd0.00e+00_drop0.0_pool7_ptypeavg_hidden128_worker3_trial7-045-0.650-0.628.pth",
        "kwargs": {"model_name": "spatial_pool_mlp", "pool_size": 7, "pool_type": "avg", "hidden_dim": 128}
    },
    "pooled_mlp_128_64": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-run_embedding_3_GAP_pooled_mlp_128_64_lr1.90e-05_wd0.00e+00_drop0.0_hidden128_64_worker0_trial0-018-0.601-0.586.pth",
        "kwargs": {"model_name": "pooled_mlp_128_64"}
    },
    "conv_gap": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-run_embedding_3_GAP_conv_gap_lr5.99e-05_wd0.00e+00_drop0.0_channels64_extraFalse_worker1_trial4-052-0.600-0.583.pth",
        "kwargs": {"model_name": "conv_gap", "cnn_channels": 64, "extra_conv": False}
    }
}

rej_results = []
f_small = models["f_small"]
f_small.eval()

try:
    data_input = f_small.extract_model_inputs(make_batch_slice(base_batch, 1))
    if isinstance(data_input, tuple): 
        data_input = data_input[0] 
except AttributeError:
    tensor_keys = [k for k, v in batch.items() if torch.is_tensor(v)]
    if "data" in tensor_keys:
        data_input = batch["data"]
    else:
        data_input = batch[tensor_keys[0]]

# Wir garantieren Batch_Size=1 für das FPGA Deployment
try:
    data_input = data_input[0:1]
except Exception:
    pass

if data_input.dim() == 3:
    data_input = data_input.unsqueeze(1) if data_input.shape[0] == 1 else data_input.unsqueeze(0)

# Sicherstellen, dass die Daten auf der GPU sind, um `Input type and weight type should be the same` zu vermeiden
data_input = data_input.to(device)

with torch.no_grad():
    feat_classifier = getattr(f_small, "classifier_features")(data_input)
    feat_pooled = getattr(f_small, "pooled_features")(data_input)


def get_rejector_state_dict(ckpt):
    state_dict = extract_state_dict(ckpt)
    clean_dict = {}
    for k, v in state_dict.items():
        if k.startswith("1.net."):
            clean_dict[k.replace("1.net.", "net.")] = v
        elif k.startswith("1."):
            clean_dict[k.replace("1.", "")] = v
        elif k.startswith("embedding_processing."):
            clean_dict[k.replace("embedding_processing.", "")] = v
            
    if len(clean_dict) == 0:
        return state_dict
    return clean_dict


for rej_name, cfg in rejector_checkpoints.items():    
    rej_model, hook = build_embedding_processing(in_channels=12, **cfg["kwargs"])
    rej_model.to(device)
    rej_model.eval()
    
    ckpt = torch.load(cfg["path"], map_location=device)
    rej_dict = get_rejector_state_dict(ckpt)
    rej_model.load_state_dict(rej_dict)
    
    if hook == "classifier_features":
        rej_input = feat_classifier.clone() # Wichtig, dass wir klonen
    elif hook == "pooled_features":
        rej_input = feat_pooled.clone()
    else:
        raise ValueError(f"Unknown hook {hook}")

    # FORCE DIMENSION CHECK FÜR DEN FLATTEN ERROR
    if rej_input.dim() == 3:
        rej_input = rej_input.unsqueeze(0)
    elif hook == "pooled_features" and rej_input.dim() == 1:
        rej_input = rej_input.unsqueeze(0)

    latency = get_latency(rej_model, rej_input)
    mem_peak = get_memory(rej_model, rej_input)
    mem_param = param_memory(rej_model)

    rej_results.append({
        "Rejector Model": rej_name,
        "Hooked Layer": hook,
        "Latency (ms)": f"{latency:.2f}",
        "Peak Mem (MB)": f"{mem_peak:.2f}", 
        "Param Mem (MB)": f"{mem_param:.4f}"
    })

pd.DataFrame(rej_results)

,Rejector Model,Hooked Layer,Latency (ms),Peak Mem (MB),Param Mem (MB)
0,conv_mlp,classifier_features,0.22,57.07,0.7930
1,spatial_pool_mlp,classifier_features,0.16,54.23,0.2886
2,pooled_mlp_128_64,pooled_features,0.16,53.08,0.0383
3,conv_gap,classifier_features,0.15,53.97,0.0271


conv_mlp		        1 * 0.21 + 1 * 0.25 + 0.75 * 0.13 = 0.5575

spatial_pool_mlp		1 * 0.12 + 1 * 0.25 + 0.75 * 0.13 = 0.4675

pooled_mlp_128_64		1 * 0.13 + 1 * 0.30 + 0.75 * 0.08 = 0.49

conv_gap		        1 * 0.15 + 1 * 0.25 + 0.75 * 0.13 = 0.498

# standard rejector

3_GAP	0.35 + 0.75 * 0.35 = 0.612 

In [ ]:
1 * 0.21 + 1 * 0.25 + x * 0.13 = y

# embedding rejector r2

In [27]:
from embedding_processing_models import build_embedding_processing

rejector_checkpoints_r2 = {
    "conv_mlp": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker13_trial6-035-0.825-0.842.pth",
        "kwargs": {"model_name": "conv_mlp", "cnn_channels": 64, "extra_conv": True, "pool_size": 7, "hidden_dim": 128}
    },
    "resnet_small_128": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-run_embedding_r2_resnet_small_lr2.23e-05_wd0.00e+00_drop0.2_channels128_extraTrue_worker2_trial5-046-0.861-0.851.pth",
        "kwargs": {"model_name": "resnet_small", "cnn_channels": 128, "extra_conv": True}
    },
    "resnet_small_64": {
        "path": "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-run_embedding_r2_resnet_small_lr1.45e-05_wd0.00e+00_drop0.2_channels64_extraTrue_worker2_trial7-077-0.822-0.847.pth",
        "kwargs": {"model_name": "resnet_small", "cnn_channels": 64, "extra_conv": True, "pool_size": 3, "hidden_dim": 128}
    }
}

rej_results_r2 = []
f_mid = models["f_mid"]
f_mid.eval()

try:
    data_input_r2 = f_mid.extract_model_inputs(make_batch_slice(base_batch, 1))
    if isinstance(data_input_r2, tuple): 
        if hasattr(f_mid, "process_inputs"):
            data_input_r2 = f_mid.process_inputs(*data_input_r2)
        else:
            data_input_r2 = data_input_r2[0] if len(data_input_r2) == 1 else torch.cat(data_input_r2, dim=1)
except AttributeError:
    tensor_keys = [k for k, v in batch.items() if torch.is_tensor(v)]
    if "data" in tensor_keys:
        data_input_r2 = batch["data"]
    else:
        data_input_r2 = batch[tensor_keys[0]]

if hasattr(f_mid, 'input_channels') and data_input_r2.shape[1] != f_mid.input_channels:
    data_input_r2 = torch.rand((1, f_mid.input_channels, 256, 256)).to(device)

try:
    data_input_r2 = data_input_r2[0:1]
except Exception:
    pass

if data_input_r2.dim() == 3:
    data_input_r2 = data_input_r2.unsqueeze(1) if data_input_r2.shape[0] == 1 else data_input_r2.unsqueeze(0)

data_input_r2 = data_input_r2.to(device)

with torch.no_grad():
    feat_classifier_r2 = getattr(f_mid, "classifier_features")(data_input_r2)
    feat_pooled_r2 = getattr(f_mid, "pooled_features")(data_input_r2)

in_channels_r2 = f_mid.out_features

def get_rejector_state_dict_fixed(ckpt):
    state_dict = extract_state_dict(ckpt)
    clean_dict = {}
    for k, v in state_dict.items():
        if k.startswith("1.net."):
            clean_dict[k.replace("1.net.", "net.")] = v
        elif k.startswith("1."):
            clean_dict[k[2:]] = v
        elif k.startswith("embedding_processing."):
            clean_dict[k.replace("embedding_processing.", "")] = v
            
    if len(clean_dict) == 0:
        return state_dict
    return clean_dict

for rej_name, cfg in rejector_checkpoints_r2.items():    
    rej_model, hook = build_embedding_processing(in_channels=in_channels_r2, **cfg["kwargs"])
    rej_model.to(device)
    rej_model.eval()
    
    ckpt = torch.load(cfg["path"], map_location=device)
    rej_dict = get_rejector_state_dict_fixed(ckpt)
    rej_model.load_state_dict(rej_dict)
    
    if hook == "classifier_features":
        rej_input = feat_classifier_r2.clone()
    elif hook == "pooled_features":
        rej_input = feat_pooled_r2.clone()
    else:
        raise ValueError(f"Unknown hook {hook}")

    if rej_input.dim() == 3:
        rej_input = rej_input.unsqueeze(0)
    elif hook == "pooled_features" and rej_input.dim() == 1:
        rej_input = rej_input.unsqueeze(0)

    latency = get_latency(rej_model, rej_input)
    mem_peak = get_memory(rej_model, rej_input)
    mem_param = param_memory(rej_model)

    rej_results_r2.append({
        "Rejector Model": rej_name,
        "Hooked Layer": hook,
        "Latency (ms)": f"{latency:.2f}",
        "Peak Mem (MB)": f"{mem_peak:.2f}", 
        "Param Mem (MB)": f"{mem_param:.4f}"
    })

pd.DataFrame(rej_results_r2)

,Rejector Model,Hooked Layer,Latency (ms),Peak Mem (MB),Param Mem (MB)
0,conv_mlp,classifier_features,0.32,91.72,1.7002
1,resnet_small_128,classifier_features,0.92,135.91,11.0029
2,resnet_small_64,classifier_features,0.92,92.54,1.7671
